# 第 4 章（Google ADK 版）

原书是 34 行**片段**（无 Runner、无法单独运行），下面补齐了驱动部分。

kernel 必须选 **`Python (agentic-adk)`**。模型换成 `gemini-3.5-flash`（`gemini-2.0-flash` 已下线）。

In [1]:
from google.adk.agents import SequentialAgent, LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
import uuid

MODEL = "gemini-3.5-flash"          # 【适配】原 gemini-2.0-flash 已下线

# The first agent generates the initial draft.
generator = LlmAgent(
    name="DraftWriter",
    model=MODEL,
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text"         # The output is saved to this state key.
)

# The second agent critiques the draft from the first agent.
reviewer = LlmAgent(
    name="FactChecker",
    model=MODEL,
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    # 【适配】原书只写了 "Read the text provided in the state key 'draft_text'"，
    #        那是文字描述，不是占位符 —— ADK 只替换 {变量名}，草稿不会进 prompt。
    instruction="""
    You are a meticulous fact-checker.
    Here is the draft to review:
    {draft_text}
    Carefully verify the factual accuracy of all claims.
    Your final output must be a dictionary containing two keys:
      - "status": A string, either "ACCURATE" or "INACCURATE".
      - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output"      # The structured dictionary is saved here.
)

# The SequentialAgent ensures the generator runs before the reviewer.
# 注：2.13 起 SequentialAgent 已标记弃用（官方让换 Workflow，但 Workflow 目前不能当 LlmAgent 的子 agent），仍可用。
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)


# --- 运行（原书缺这一段）---
def run_pipeline(pipeline, subject: str):
    runner = InMemoryRunner(pipeline)
    session_id = str(uuid.uuid4())

    # 【适配】create_session 在 ADK 2.13 是协程，同步上下文用 create_session_sync
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="user_1", session_id=session_id
    )

    # 注意：SequentialAgent 里【每个子 agent】都会产生 final response，
    #      所以这里收全部，而不是 break 在第一个（那样只会拿到草稿）。
    finals = []
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=subject)]),
    ):
        if event.is_final_response() and event.content:
            text = (
                event.content.text
                if getattr(event.content, "text", None)
                else "".join(p.text for p in (event.content.parts or []) if getattr(p, "text", None))
            )
            finals.append((getattr(event, "author", "?"), text))
    return finals


for author, text in run_pipeline(review_pipeline, "The Eiffel Tower"):
    print(f"\n--- [{author}] ---")
    print(text[:600])

/tmp/ipykernel_2949220/571516370.py:38: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  review_pipeline = SequentialAgent(
Deprecated. Please migrate to the async method.



--- [DraftWriter] ---
The Eiffel Tower, located on the Champ de Mars in Paris, France, is one of the world’s most recognizable landmarks. Designed by civil engineer Gustave Eiffel and completed in 1889, the wrought-iron lattice tower was constructed as the entrance arch for the Exposition Universelle (World's Fair) to celebrate the centennial of the French Revolution. Standing at 330 meters (1,083 feet) tall including its tip, it held the title of the world's tallest man-made structure for over 40 years. Today, the Eiffel Tower is a global cultural symbol of France, attracting millions of visitors each year as one 

--- [FactChecker] ---
```json
{
  "status": "ACCURATE",
  "reasoning": "All claims in the draft were verified for factual accuracy: 1) The Eiffel Tower is located on the Champ de Mars in Paris, France. 2) It was designed by civil engineer Gustave Eiffel (and his firm) and completed in 1889 as the entrance arch for the 1889 Exposition Universelle celebrating the centennial 